<a href="https://colab.research.google.com/github/OMatheusWander/CienciadeDadosUFSC/blob/Algebra_Linear_para_Ciencia_de_Dados/%C3%81lgebra_Linear.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

TOLERANCIA = 1e-10  # valores menores que isso são tratados como zero

## 0. Funções auxiliares

In [ ]:
def dimensoes(matriz):
    """Retorna (linhas, colunas) da matriz."""
    linhas = matriz.shape[0]
    colunas = matriz.shape[1]
    return linhas, colunas


def e_quadrada(matriz):
    """Verifica se a matriz é quadrada."""
    linhas, colunas = dimensoes(matriz)
    if linhas == colunas:
        return True
    else:
        return False


def limpar_zeros(matriz):
    """Substitui valores muito próximos de zero por 0 (evita -0.0 e 1e-17)."""
    resultado = matriz.copy()
    linhas, colunas = dimensoes(resultado)
    for i in range(linhas):
        for j in range(colunas):
            if abs(resultado[i, j]) < TOLERANCIA:
                resultado[i, j] = 0.0
    return resultado


def trocar_linhas(matriz, linha_a, linha_b):
    """Troca duas linhas da matriz (altera a matriz recebida)."""
    colunas = matriz.shape[1]
    for j in range(colunas):
        temporario = matriz[linha_a, j]
        matriz[linha_a, j] = matriz[linha_b, j]
        matriz[linha_b, j] = temporario


def imprimir_matriz(matriz, titulo=""):
    """Imprime a matriz de forma legível."""
    if titulo != "":
        print(titulo)
    linhas, colunas = dimensoes(matriz)
    for i in range(linhas):
        texto_linha = ""
        for j in range(colunas):
            texto_linha = texto_linha + f"{matriz[i, j]:10.4f}"
        print(texto_linha)
    print()

## 1. Soma e subtração de matrizes

$C_{ij} = A_{ij} + B_{ij}$ — exige mesmas dimensões.

In [ ]:
def soma_matrizes(A, B):
    """Soma elemento a elemento de duas matrizes de mesma dimensão."""
    linhas_a, colunas_a = dimensoes(A)
    linhas_b, colunas_b = dimensoes(B)

    if linhas_a != linhas_b or colunas_a != colunas_b:
        raise ValueError("As matrizes precisam ter as mesmas dimensões para a soma.")

    C = np.zeros((linhas_a, colunas_a))
    for i in range(linhas_a):
        for j in range(colunas_a):
            C[i, j] = A[i, j] + B[i, j]
    return C


def subtracao_matrizes(A, B):
    """Subtração elemento a elemento: A - B."""
    linhas_a, colunas_a = dimensoes(A)
    linhas_b, colunas_b = dimensoes(B)

    if linhas_a != linhas_b or colunas_a != colunas_b:
        raise ValueError("As matrizes precisam ter as mesmas dimensões para a subtração.")

    C = np.zeros((linhas_a, colunas_a))
    for i in range(linhas_a):
        for j in range(colunas_a):
            C[i, j] = A[i, j] - B[i, j]
    return C

In [ ]:
A = np.array([[1, 2], [3, 4]], dtype=float)
B = np.array([[5, 6], [7, 8]], dtype=float)

imprimir_matriz(soma_matrizes(A, B), "A + B:")
imprimir_matriz(subtracao_matrizes(A, B), "A - B:")
print("Conferência numpy:", np.allclose(soma_matrizes(A, B), A + B))

## 2. Multiplicação por escalar

$C_{ij} = k \cdot A_{ij}$

In [ ]:
def multiplicacao_escalar(k, A):
    """Multiplica cada elemento da matriz pelo escalar k."""
    linhas, colunas = dimensoes(A)
    C = np.zeros((linhas, colunas))
    for i in range(linhas):
        for j in range(colunas):
            C[i, j] = k * A[i, j]
    return C

In [ ]:
imprimir_matriz(multiplicacao_escalar(3, A), "3 * A:")

## 3. Multiplicação de matrizes

$C_{ij} = \sum_{k=1}^{n} A_{ik} \cdot B_{kj}$ — o nº de colunas de A deve ser igual ao nº de linhas de B.

In [ ]:
def multiplicacao_matrizes(A, B):
    """Produto matricial A x B (linha de A por coluna de B)."""
    linhas_a, colunas_a = dimensoes(A)
    linhas_b, colunas_b = dimensoes(B)

    if colunas_a != linhas_b:
        raise ValueError("Nº de colunas de A deve ser igual ao nº de linhas de B.")

    C = np.zeros((linhas_a, colunas_b))
    for i in range(linhas_a):
        for j in range(colunas_b):
            soma = 0.0
            for k in range(colunas_a):
                soma = soma + A[i, k] * B[k, j]
            C[i, j] = soma
    return C

In [ ]:
M = np.array([[1, 2, 3], [4, 5, 6]], dtype=float)   # 2x3
N = np.array([[7, 8], [9, 10], [11, 12]], dtype=float)  # 3x2

imprimir_matriz(multiplicacao_matrizes(M, N), "M x N (2x2):")
print("Conferência numpy:", np.allclose(multiplicacao_matrizes(M, N), M @ N))

## 4. Transposta

$A^T_{ij} = A_{ji}$

In [ ]:
def transposta(A):
    """Troca linhas por colunas."""
    linhas, colunas = dimensoes(A)
    T = np.zeros((colunas, linhas))
    for i in range(linhas):
        for j in range(colunas):
            T[j, i] = A[i, j]
    return T

In [ ]:
imprimir_matriz(M, "M:")
imprimir_matriz(transposta(M), "Transposta de M:")
print("Conferência numpy:", np.allclose(transposta(M), M.T))

## 5. Triangularização por eliminação de Gauss

Transforma a matriz em **triangular superior** zerando os elementos abaixo da diagonal principal.
Usa pivoteamento parcial (escolhe o maior pivô em módulo) para estabilidade numérica.
Retorna também o número de trocas de linha — necessário para o sinal do determinante.

In [ ]:
def triangular_superior(A, mostrar_passos=False):
    """
    Escalona A até a forma triangular superior (eliminação de Gauss).
    Retorna: (matriz_triangular, numero_de_trocas)
    """
    U = A.astype(float).copy()
    linhas, colunas = dimensoes(U)
    numero_trocas = 0

    linha_pivo = 0
    for coluna in range(colunas):
        if linha_pivo >= linhas:
            break

        # 1) Pivoteamento parcial: procura o maior valor absoluto na coluna
        indice_maior = linha_pivo
        maior_valor = abs(U[linha_pivo, coluna])
        for i in range(linha_pivo + 1, linhas):
            if abs(U[i, coluna]) > maior_valor:
                maior_valor = abs(U[i, coluna])
                indice_maior = i

        # 2) Coluna sem pivô válido: passa para a próxima coluna
        if maior_valor < TOLERANCIA:
            continue

        # 3) Troca de linhas, se necessário
        if indice_maior != linha_pivo:
            trocar_linhas(U, linha_pivo, indice_maior)
            numero_trocas = numero_trocas + 1
            if mostrar_passos:
                print(f"L{linha_pivo + 1} <-> L{indice_maior + 1}")

        # 4) Zera os elementos abaixo do pivô: Li = Li - fator * Lpivo
        for i in range(linha_pivo + 1, linhas):
            fator = U[i, coluna] / U[linha_pivo, coluna]
            if abs(fator) > TOLERANCIA:
                for j in range(colunas):
                    U[i, j] = U[i, j] - fator * U[linha_pivo, j]
                if mostrar_passos:
                    print(f"L{i + 1} = L{i + 1} - ({fator:.4f}) * L{linha_pivo + 1}")

        if mostrar_passos:
            imprimir_matriz(limpar_zeros(U))

        linha_pivo = linha_pivo + 1

    return limpar_zeros(U), numero_trocas

In [ ]:
Q = np.array([[2, 1, -1],
              [-3, -1, 2],
              [-2, 1, 2]], dtype=float)

U, trocas = triangular_superior(Q, mostrar_passos=True)
imprimir_matriz(U, f"Triangular superior (trocas de linha: {trocas}):")

## 6. Determinante

**Método 1 — Triangulação:** $\det(A) = (-1)^{trocas} \cdot \prod U_{ii}$

**Método 2 — Laplace (cofatores):** $\det(A) = \sum_{j} (-1)^{1+j} \cdot a_{1j} \cdot \det(M_{1j})$ — didático, mas lento para matrizes grandes.

In [ ]:
def determinante(A):
    """Determinante via triangularização de Gauss."""
    if not e_quadrada(A):
        raise ValueError("O determinante só existe para matrizes quadradas.")

    U, numero_trocas = triangular_superior(A)
    n = U.shape[0]

    produto_diagonal = 1.0
    for i in range(n):
        produto_diagonal = produto_diagonal * U[i, i]

    if numero_trocas % 2 == 0:
        sinal = 1
    else:
        sinal = -1

    resultado = sinal * produto_diagonal
    if abs(resultado) < TOLERANCIA:
        resultado = 0.0
    return resultado


def submatriz_menor(A, linha_removida, coluna_removida):
    """Retorna a matriz sem a linha e a coluna indicadas (menor complementar)."""
    n = A.shape[0]
    menor = np.zeros((n - 1, n - 1))
    nova_i = 0
    for i in range(n):
        if i == linha_removida:
            continue
        nova_j = 0
        for j in range(n):
            if j == coluna_removida:
                continue
            menor[nova_i, nova_j] = A[i, j]
            nova_j = nova_j + 1
        nova_i = nova_i + 1
    return menor


def determinante_laplace(A):
    """Determinante via expansão de Laplace pela primeira linha (recursivo)."""
    if not e_quadrada(A):
        raise ValueError("O determinante só existe para matrizes quadradas.")

    n = A.shape[0]

    if n == 1:
        return A[0, 0]

    if n == 2:
        return A[0, 0] * A[1, 1] - A[0, 1] * A[1, 0]

    soma = 0.0
    for j in range(n):
        if j % 2 == 0:
            sinal = 1
        else:
            sinal = -1
        menor = submatriz_menor(A, 0, j)
        soma = soma + sinal * A[0, j] * determinante_laplace(menor)
    return soma

In [ ]:
print("det(Q) por triangulação:", determinante(Q))
print("det(Q) por Laplace:     ", determinante_laplace(Q))
print("det(Q) numpy:           ", np.linalg.det(Q))

singular = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]], dtype=float)
print("\ndet(matriz singular):", determinante(singular))

## 7. Escalonamento completo de sistemas (Gauss-Jordan)

Leva a matriz aumentada $[A \mid b]$ à **forma escalonada reduzida por linhas** (pivôs = 1 e zeros acima e abaixo).

Classificação pelo posto (Rouché-Capelli):

| Condição | Classificação |
|---|---|
| posto(A) ≠ posto([A\|b]) | SI — Sistema Impossível |
| posto(A) = posto([A\|b]) = nº de incógnitas | SPD — Possível e Determinado |
| posto(A) = posto([A\|b]) < nº de incógnitas | SPI — Possível e Indeterminado |

In [ ]:
def forma_escalonada_reduzida(A, mostrar_passos=False, colunas_pivo_limite=None):
    """
    Gauss-Jordan: retorna (matriz_reduzida, lista_colunas_pivo).
    colunas_pivo_limite: procura pivôs só até essa coluna (útil para a matriz aumentada,
    em que a última coluna é o vetor b e não deve virar pivô).
    """
    R = A.astype(float).copy()
    linhas, colunas = dimensoes(R)

    if colunas_pivo_limite is None:
        colunas_pivo_limite = colunas

    colunas_pivo = []
    linha_pivo = 0

    for coluna in range(colunas_pivo_limite):
        if linha_pivo >= linhas:
            break

        # 1) Pivoteamento parcial
        indice_maior = linha_pivo
        maior_valor = abs(R[linha_pivo, coluna])
        for i in range(linha_pivo + 1, linhas):
            if abs(R[i, coluna]) > maior_valor:
                maior_valor = abs(R[i, coluna])
                indice_maior = i

        if maior_valor < TOLERANCIA:
            continue

        if indice_maior != linha_pivo:
            trocar_linhas(R, linha_pivo, indice_maior)
            if mostrar_passos:
                print(f"L{linha_pivo + 1} <-> L{indice_maior + 1}")

        # 2) Normaliza a linha do pivô: Lpivo = Lpivo / pivo
        pivo = R[linha_pivo, coluna]
        for j in range(colunas):
            R[linha_pivo, j] = R[linha_pivo, j] / pivo
        if mostrar_passos:
            print(f"L{linha_pivo + 1} = L{linha_pivo + 1} / ({pivo:.4f})")

        # 3) Zera a coluna do pivô em TODAS as outras linhas (acima e abaixo)
        for i in range(linhas):
            if i == linha_pivo:
                continue
            fator = R[i, coluna]
            if abs(fator) > TOLERANCIA:
                for j in range(colunas):
                    R[i, j] = R[i, j] - fator * R[linha_pivo, j]
                if mostrar_passos:
                    print(f"L{i + 1} = L{i + 1} - ({fator:.4f}) * L{linha_pivo + 1}")

        if mostrar_passos:
            imprimir_matriz(limpar_zeros(R))

        colunas_pivo.append(coluna)
        linha_pivo = linha_pivo + 1

    return limpar_zeros(R), colunas_pivo


def posto(matriz):
    """Posto = número de linhas não nulas após escalonamento."""
    R, colunas_pivo = forma_escalonada_reduzida(matriz)
    linhas, colunas = dimensoes(R)
    contador = 0
    for i in range(linhas):
        linha_nula = True
        for j in range(colunas):
            if abs(R[i, j]) > TOLERANCIA:
                linha_nula = False
                break
        if not linha_nula:
            contador = contador + 1
    return contador


def resolver_sistema(A, b, mostrar_passos=False):
    """
    Resolve Ax = b por Gauss-Jordan na matriz aumentada.
    Retorna um dicionário com: classificacao, matriz_reduzida, solucao, variaveis_livres.
    """
    linhas, n_incognitas = dimensoes(A)
    b_coluna = b.astype(float).reshape(linhas, 1)

    # Monta a matriz aumentada [A | b]
    aumentada = np.zeros((linhas, n_incognitas + 1))
    for i in range(linhas):
        for j in range(n_incognitas):
            aumentada[i, j] = A[i, j]
        aumentada[i, n_incognitas] = b_coluna[i, 0]

    if mostrar_passos:
        imprimir_matriz(aumentada, "Matriz aumentada [A | b]:")

    R, colunas_pivo = forma_escalonada_reduzida(aumentada, mostrar_passos, colunas_pivo_limite=n_incognitas)

    posto_a = posto(A)
    posto_aumentada = posto(aumentada)

    resultado = {
        "matriz_reduzida": R,
        "posto_A": posto_a,
        "posto_aumentada": posto_aumentada,
        "solucao": None,
        "variaveis_livres": [],
    }

    if posto_a != posto_aumentada:
        resultado["classificacao"] = "SI - Sistema Impossível"
        return resultado

    if posto_a == n_incognitas:
        resultado["classificacao"] = "SPD - Sistema Possível e Determinado"
        solucao = np.zeros(n_incognitas)
        for indice_linha in range(len(colunas_pivo)):
            coluna = colunas_pivo[indice_linha]
            solucao[coluna] = R[indice_linha, n_incognitas]
        resultado["solucao"] = solucao
        return resultado

    # SPI: descreve cada variável pivô em função das livres
    resultado["classificacao"] = "SPI - Sistema Possível e Indeterminado"
    variaveis_livres = []
    for j in range(n_incognitas):
        if j not in colunas_pivo:
            variaveis_livres.append(j)
    resultado["variaveis_livres"] = variaveis_livres

    expressoes = []
    for indice_linha in range(len(colunas_pivo)):
        coluna = colunas_pivo[indice_linha]
        texto = f"x{coluna + 1} = {R[indice_linha, n_incognitas]:.4f}"
        for livre in variaveis_livres:
            coeficiente = -R[indice_linha, livre]
            if abs(coeficiente) > TOLERANCIA:
                if coeficiente > 0:
                    texto = texto + f" + {coeficiente:.4f}*x{livre + 1}"
                else:
                    texto = texto + f" - {abs(coeficiente):.4f}*x{livre + 1}"
        expressoes.append(texto)
    for livre in variaveis_livres:
        expressoes.append(f"x{livre + 1} livre")
    resultado["solucao"] = expressoes
    return resultado

In [ ]:
# Exemplo SPD
#  2x +  y -  z =   8
# -3x -  y + 2z = -11
# -2x +  y + 2z =  -3
b = np.array([8, -11, -3], dtype=float)
res = resolver_sistema(Q, b, mostrar_passos=True)
print(res["classificacao"])
print("Solução:", res["solucao"])

In [ ]:
# Exemplo SPI
A_spi = np.array([[1, 2, 3], [2, 4, 6], [1, 1, 1]], dtype=float)
b_spi = np.array([6, 12, 3], dtype=float)
res = resolver_sistema(A_spi, b_spi)
print(res["classificacao"])
for expressao in res["solucao"]:
    print("  ", expressao)

# Exemplo SI
A_si = np.array([[1, 1], [1, 1]], dtype=float)
b_si = np.array([2, 3], dtype=float)
print("\n" + resolver_sistema(A_si, b_si)["classificacao"])

## 8. Matriz cofator e matriz adjunta

$C_{ij} = (-1)^{i+j} \cdot \det(M_{ij})$, onde $M_{ij}$ é a matriz sem a linha $i$ e a coluna $j$.

Adjunta: $\text{adj}(A) = C^T$

In [ ]:
def matriz_cofator(A):
    """Calcula a matriz dos cofatores de A."""
    if not e_quadrada(A):
        raise ValueError("A matriz cofator só existe para matrizes quadradas.")

    n = A.shape[0]
    C = np.zeros((n, n))

    if n == 1:
        C[0, 0] = 1.0
        return C

    for i in range(n):
        for j in range(n):
            menor = submatriz_menor(A, i, j)
            if (i + j) % 2 == 0:
                sinal = 1
            else:
                sinal = -1
            C[i, j] = sinal * determinante(menor)
    return limpar_zeros(C)


def matriz_adjunta(A):
    """Adjunta = transposta da matriz cofator."""
    return transposta(matriz_cofator(A))

In [ ]:
imprimir_matriz(matriz_cofator(Q), "Cofatores de Q:")
imprimir_matriz(matriz_adjunta(Q), "Adjunta de Q:")

## 9. Matriz inversa

**Método 1 — Adjunta:** $A^{-1} = \dfrac{1}{\det(A)} \cdot \text{adj}(A)$

**Método 2 — Gauss-Jordan:** escalona $[A \mid I]$ até $[I \mid A^{-1}]$ (mais eficiente).

Só existe se $\det(A) \neq 0$.

In [ ]:
def inversa_por_adjunta(A):
    """Inversa via fórmula da adjunta."""
    if not e_quadrada(A):
        raise ValueError("Só matrizes quadradas podem ter inversa.")

    det = determinante(A)
    if abs(det) < TOLERANCIA:
        raise ValueError("Matriz singular (det = 0): não possui inversa.")

    adj = matriz_adjunta(A)
    return multiplicacao_escalar(1.0 / det, adj)


def matriz_identidade(n):
    """Cria a matriz identidade n x n."""
    I = np.zeros((n, n))
    for i in range(n):
        I[i, i] = 1.0
    return I


def inversa_gauss_jordan(A, mostrar_passos=False):
    """Inversa escalonando [A | I] até [I | A^-1]."""
    if not e_quadrada(A):
        raise ValueError("Só matrizes quadradas podem ter inversa.")

    n = A.shape[0]
    I = matriz_identidade(n)

    # Monta [A | I]
    aumentada = np.zeros((n, 2 * n))
    for i in range(n):
        for j in range(n):
            aumentada[i, j] = A[i, j]
            aumentada[i, n + j] = I[i, j]

    R, colunas_pivo = forma_escalonada_reduzida(aumentada, mostrar_passos, colunas_pivo_limite=n)

    if len(colunas_pivo) < n:
        raise ValueError("Matriz singular: não possui inversa.")

    inversa = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            inversa[i, j] = R[i, n + j]
    return limpar_zeros(inversa)

In [ ]:
inv_adj = inversa_por_adjunta(Q)
inv_gj = inversa_gauss_jordan(Q)

imprimir_matriz(inv_adj, "Inversa de Q (adjunta):")
imprimir_matriz(inv_gj, "Inversa de Q (Gauss-Jordan):")
imprimir_matriz(limpar_zeros(multiplicacao_matrizes(Q, inv_gj)), "Prova: Q x Q^-1 = I")
print("Conferência numpy:", np.allclose(inv_gj, np.linalg.inv(Q)))

try:
    inversa_gauss_jordan(singular)
except ValueError as erro:
    print("\nMatriz singular ->", erro)

## 10. Validação geral contra o numpy

In [ ]:
np.random.seed(42)
todos_ok = True

for teste in range(20):
    n = np.random.randint(2, 6)
    X = np.random.randint(-9, 10, size=(n, n)).astype(float)
    Y = np.random.randint(-9, 10, size=(n, n)).astype(float)

    checagens = {
        "soma": np.allclose(soma_matrizes(X, Y), X + Y),
        "produto": np.allclose(multiplicacao_matrizes(X, Y), X @ Y),
        "transposta": np.allclose(transposta(X), X.T),
        "determinante": np.isclose(determinante(X), np.linalg.det(X)),
    }

    if abs(np.linalg.det(X)) > 1e-6:
        checagens["inversa_adj"] = np.allclose(inversa_por_adjunta(X), np.linalg.inv(X))
        checagens["inversa_gj"] = np.allclose(inversa_gauss_jordan(X), np.linalg.inv(X))
        b_teste = np.random.randint(-9, 10, size=n).astype(float)
        checagens["sistema"] = np.allclose(resolver_sistema(X, b_teste)["solucao"], np.linalg.solve(X, b_teste))

    for nome in checagens:
        if not checagens[nome]:
            todos_ok = False
            print(f"Falha no teste {teste}: {nome}")

if todos_ok:
    print("Todas as funções conferem com o numpy em 20 matrizes aleatórias.")